# Crop Yield Prediction in Kenya (2000–2023)

## Exploratory Data Analysis (EDA)

This notebook performs the initial exploration of the datasets used to develop a machine learning model for predicting crop yield in Kenya.

### Objectives

- Load all datasets
- Inspect their structure and contents
- Assess data quality by checking for missing values and duplicates
- Prepare the datasets for further preprocessing and modelling

The datasets used in this project are:

1. FAOSTAT agricultural statistics
2. NDVI (Normalized Difference Vegetation Index)
3. Annual Rainfall
4. Annual Land Surface Temperature

In [21]:
# Import Required Libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Display all dataframe columns
pd.set_option("display.max_columns", None)

# Display wider tables
pd.set_option("display.width", 1000)

## Load the Datasets

The raw datasets are stored in the `Data/Raw_data` folder and are loaded into pandas DataFrames for analysis.

In [22]:
# Load Raw Datasets

faostat_df = pd.read_csv("../Data/Raw_data/Faostat.csv")

ndvi_df = pd.read_csv("../Data/Raw_data/NDVI.csv")

rainfall_df = pd.read_csv("../Data/Raw_data/Rainfall.csv")

temperature_df = pd.read_csv("../Data/Raw_data/Temperature.csv")

print("All datasets loaded successfully.")

All datasets loaded successfully.


## Dataset Inspection

This section examines the structure and contents of each dataset. We inspect the dimensions, column names, data types, and the first few records to understand the data before performing any cleaning or transformation.

In [26]:
# Inspect the Datasets

datasets = {
    "FAOSTAT": faostat_df,
    "NDVI": ndvi_df,
    "Rainfall": rainfall_df,
    "Temperature": temperature_df
}

for dataset_name, dataset_df in datasets.items():

    print("=" * 20)
    print(dataset_name)
    print("=" * 20)

    print(f"Shape: {dataset_df.shape}")

    print("\nColumns:")
    print(dataset_df.columns.tolist())

    print("\nData Types:")
    print(dataset_df.dtypes)

    print("\nFirst Five Rows:")
    display(dataset_df.head())

    print("\n")

FAOSTAT
Shape: (375, 5)

Columns:
['Item', 'Element', 'Year', 'Unit', 'Value']

Data Types:
Item           str
Element        str
Year         int64
Unit           str
Value      float64
dtype: object

First Five Rows:


,Item,Element,Year,Unit,Value
0,Rice,Area harvested,2000,ha,13882.0
1,Rice,Yield,2000,kg/ha,3771.0
2,Rice,Production,2000,t,52349.0
3,Rice,Area harvested,2001,ha,13200.0
4,Rice,Yield,2001,kg/ha,3409.1




NDVI
Shape: (120, 3)

Columns:
['Crop', 'Weighted_Mean_NDVI', 'Year']

Data Types:
Crop                      str
Weighted_Mean_NDVI    float64
Year                    int64
dtype: object

First Five Rows:


,Crop,Weighted_Mean_NDVI,Year
0,Maize,0.518891,2000
1,Maize,0.567526,2001
2,Maize,0.567220,2002
3,Maize,0.573118,2003
4,Maize,0.559780,2004




Rainfall
Shape: (120, 3)

Columns:
['Crop', 'Rainfall_mm', 'Year']

Data Types:
Crop               str
Rainfall_mm    float64
Year             int64
dtype: object

First Five Rows:


,Crop,Rainfall_mm,Year
0,Maize,991.254323,2000
1,Maize,1241.764138,2001
2,Maize,1267.337974,2002
3,Maize,1135.643418,2003
4,Maize,1095.686872,2004




Temperature
Shape: (120, 3)

Columns:
['Crop', 'Weighted_Mean_Temperature_C', 'Year']

Data Types:
Crop                               str
Weighted_Mean_Temperature_C    float64
Year                             int64
dtype: object

First Five Rows:


,Crop,Weighted_Mean_Temperature_C,Year
0,Maize,31.868734,2000
1,Maize,30.316524,2001
2,Maize,29.997854,2002
3,Maize,30.491539,2003
4,Maize,30.388731,2004


## Data Quality Assessment

Before transforming the datasets, it is important to assess their quality. This section checks for missing values and duplicate records in each dataset to identify potential issues that could affect the analysis and model performance.

In [27]:
# Assess Data Quality

datasets = {
    "FAOSTAT": faostat_df,
    "NDVI": ndvi_df,
    "Rainfall": rainfall_df,
    "Temperature": temperature_df
}

for dataset_name, dataset_df in datasets.items():

    print("=" * 20)
    print(dataset_name)
    print("=" * 20)

    print("\nMissing Values")
    print(dataset_df.isnull().sum())

    print("\nDuplicate Records")
    print(dataset_df.duplicated().sum())

    print("\n")

FAOSTAT

Missing Values
Item       0
Element    0
Year       0
Unit       0
Value      0
dtype: int64

Duplicate Records
0


NDVI

Missing Values
Crop                  0
Weighted_Mean_NDVI    0
Year                  0
dtype: int64

Duplicate Records
0


Rainfall

Missing Values
Crop           0
Rainfall_mm    0
Year           0
dtype: int64

Duplicate Records
0


Temperature

Missing Values
Crop                           0
Weighted_Mean_Temperature_C    0
Year                           0
dtype: int64

Duplicate Records
0




## Clean and Prepare the FAOSTAT Dataset

The FAOSTAT dataset is transformed into a machine-learning-ready format.

The following preprocessing steps are performed:

- Convert production values from tonnes (t) to kilograms (kg) for consistency.
- Pivot the dataset so that each row represents one crop in one year.
- Rename the pivoted columns to descriptive names.
- Arrange the dataset by Year and Crop to prepare it for merging with the NDVI, rainfall, and temperature datasets.

In [28]:
# Clean and Prepare the FAOSTAT Dataset

# Create a copy to preserve the original dataset
faostat_clean_df = faostat_df.copy()

# Convert production from tonnes to kilograms
production_mask = faostat_clean_df["Element"] == "Production"

faostat_clean_df.loc[production_mask, "Value"] = (
    faostat_clean_df.loc[production_mask, "Value"] * 1000
)

faostat_clean_df.loc[production_mask, "Unit"] = "kg"

# Pivot the dataset
faostat_clean_df = (
    faostat_clean_df
    .pivot(
        index=["Year", "Item"],
        columns="Element",
        values="Value"
    )
    .reset_index()
)

# Rename columns
faostat_clean_df = faostat_clean_df.rename(columns={
    "Item": "Crop",
    "Area harvested": "Harvested_Area_ha",
    "Production": "Production_kg",
    "Yield": "Yield_kg_per_ha"
})

# Arrange columns in the desired order
faostat_clean_df = faostat_clean_df[
    [
        "Year",
        "Crop",
        "Harvested_Area_ha",
        "Production_kg",
        "Yield_kg_per_ha"
    ]
]

# Sort by Year first, then Crop
faostat_clean_df = (
    faostat_clean_df
    .sort_values(
        by=["Year", "Crop"],
        ascending=[True, True]
    )
    .reset_index(drop=True)
)

print("FAOSTAT dataset prepared successfully.")

display(faostat_clean_df.head(15))

FAOSTAT dataset prepared successfully.


Element,Year,Crop,Harvested_Area_ha,Production_kg,Yield_kg_per_ha
0,2000,Maize (corn),1500000.0,2.160000e+09,1440.0
1,2000,Rice,13882.0,5.234900e+07,3771.0
2,2000,Sugar cane,57243.0,3.941524e+09,68856.0
3,2000,Tea leaves,120390.0,1.027000e+09,8530.6
4,2000,Wheat,131834.0,2.042320e+08,1549.2
5,2001,Maize (corn),1640000.0,2.790000e+09,1701.2
6,2001,Rice,13200.0,4.500000e+07,3409.1
7,2001,Sugar cane,47794.0,3.550792e+09,74293.7
8,2001,Tea leaves,124290.0,1.281000e+09,10306.5
9,2001,Wheat,129209.0,2.569970e+08,1989.0


## Standardize Crop Names

To ensure successful merging, crop names in the FAOSTAT dataset are standardized to match the NDVI, rainfall, and temperature datasets.

In [29]:
# Standardize Crop Names

crop_name_mapping = {
    "Maize (corn)": "Maize",
    "Sugar cane": "Sugarcane",
    "Tea leaves": "Tea"
}

faostat_clean_df["Crop"] = (
    faostat_clean_df["Crop"]
    .replace(crop_name_mapping)
)

print("Unique crop names:")

print(sorted(faostat_clean_df["Crop"].unique()))

Unique crop names:
['Maize', 'Rice', 'Sugarcane', 'Tea', 'Wheat']


## Merge the Datasets

The cleaned FAOSTAT dataset is merged with the NDVI, rainfall, and temperature datasets using the common keys **Crop** and **Year**. The resulting dataset will be used for exploratory data analysis and machine learning.

In [30]:
# Merge All Datasets

# Merge FAOSTAT with NDVI
merged_df = pd.merge(
    faostat_clean_df,
    ndvi_df,
    on=["Crop", "Year"],
    how="inner"
)

# Merge with Rainfall
merged_df = pd.merge(
    merged_df,
    rainfall_df,
    on=["Crop", "Year"],
    how="inner"
)

# Merge with Temperature
merged_df = pd.merge(
    merged_df,
    temperature_df,
    on=["Crop", "Year"],
    how="inner"
)

print("Datasets merged successfully.")

print(f"\nNumber of rows: {merged_df.shape[0]}")
print(f"Number of columns: {merged_df.shape[1]}")

display(merged_df.head(10))

Datasets merged successfully.

Number of rows: 120
Number of columns: 8


,Year,Crop,Harvested_Area_ha,Production_kg,Yield_kg_per_ha,Weighted_Mean_NDVI,Rainfall_mm,Weighted_Mean_Temperature_C
0,2000,Maize,1500000.0,2.160000e+09,1440.0,0.518891,991.254323,31.868734
1,2000,Rice,13882.0,5.234900e+07,3771.0,0.552471,850.924221,30.684194
2,2000,Sugarcane,57243.0,3.941524e+09,68856.0,0.597615,1355.680401,30.743833
3,2000,Tea,120390.0,1.027000e+09,8530.6,0.602310,1111.508038,28.538675
4,2000,Wheat,131834.0,2.042320e+08,1549.2,0.518453,835.072341,28.969239
5,2001,Maize,1640000.0,2.790000e+09,1701.2,0.567526,1241.764138,30.316524
6,2001,Rice,13200.0,4.500000e+07,3409.1,0.590736,1242.024533,28.883785
7,2001,Sugarcane,47794.0,3.550792e+09,74293.7,0.631959,1643.924375,29.970129
8,2001,Tea,124290.0,1.281000e+09,10306.5,0.647247,1448.844004,26.938514
9,2001,Wheat,129209.0,2.569970e+08,1989.0,0.604928,1170.410494,26.066822


## Save the Processed Dataset

The cleaned and merged dataset is saved to the `Data/Processed_data` folder for use in the exploratory data analysis, modelling, and evaluation notebooks.

In [ ]:
# Save the Processed Dataset

output_path = "../Data/Processed_data/Crop_Yield_Kenya.csv"

merged_df.to_csv(output_path, index=False)

print("Processed dataset saved successfully.")
print(f"Location: {output_path}")